# Capítulo 5 · Actividad: ¿quién sobrevivió al Titanic?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/riosinda/ml-course/blob/main/notebooks/exercises/chapter_05/01_actividad_titanic_arbol.ipynb)

Entrena un **árbol de decisión** que prediga si un pasajero sobrevivió (`survived`) y consigue el mejor modelo que puedas. Haz todo el proceso que vimos en clase; el preprocesamiento lo decides tú.

1. **Copia esta notebook a `workspace/`** y trabaja sobre la copia.
2. Si todavía no tienes los datos: `uv run python scripts/download_data.py`

In [ ]:
from pathlib import Path

import pandas as pd

# Carpeta data/ en la raíz del repo: subimos de carpeta en carpeta hasta dar con pyproject.toml
raiz = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()), Path.cwd())
DATA_DIR = raiz / "data"

df = pd.read_csv(DATA_DIR / "titanic.csv")
df.head()

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# 1. Copiar datos y pasar todos los nombres de columnas a minúsculas
df_avanzado = df.copy()
df_avanzado.columns = df_avanzado.columns.str.lower()

# 2. Ingeniería de Características
df_avanzado['familysize'] = df_avanzado['sibsp'] + df_avanzado['parch'] + 1
df_avanzado['is_alone'] = (df_avanzado['familysize'] == 1).astype(int)

# Si tiene la columna 'name' (CSV de GitHub), extraemos el título; si tiene 'who' (CSV del curso), lo usamos directo
if 'name' in df_avanzado.columns:
    df_avanzado['title'] = df_avanzado['name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
    df_avanzado['title'] = df_avanzado['title'].replace(['Ms', 'Mlle'], 'Miss')
    df_avanzado['title'] = df_avanzado['title'].replace(['Mme'], 'Mrs')
    titulos_comunes = ['Mr', 'Miss', 'Mrs', 'Master']
    df_avanzado['title'] = df_avanzado['title'].where(df_avanzado['title'].isin(titulos_comunes), 'Rare')

# Aprovechar cabina/cubierta si existe antes de limpiar
if 'cabin' in df_avanzado.columns:
    df_avanzado['has_cabin'] = df_avanzado['cabin'].notna().astype(int)
elif 'deck' in df_avanzado.columns:
    df_avanzado['has_cabin'] = df_avanzado['deck'].notna().astype(int)

# 3. Eliminar columnas irrelevantes o que causan fuga de datos (como 'alive')
cols_borrar = [
    'passengerid', 'name', 'ticket', 'cabin',
    'alive', 'class', 'embark_town', 'adult_male', 'deck', 'alone'
]
df_avanzado = df_avanzado.drop(columns=cols_borrar, errors='ignore')

# 4. Separar X e y ANTES de imputar nulos
X_avanzado = df_avanzado.drop(columns=['survived'])
y_avanzado = df_avanzado['survived']

X_train_av, X_test_av, y_train_av, y_test_av = train_test_split(
    X_avanzado, y_avanzado, test_size=0.2, random_state=42, stratify=y_avanzado
)

# 5. Imputar valores faltantes usando solo X_train_av
mediana_edad = X_train_av['age'].median()
moda_embarque = X_train_av['embarked'].mode()[0]
mediana_fare = X_train_av['fare'].median()

for dataset in [X_train_av, X_test_av]:
    dataset['age'] = dataset['age'].fillna(mediana_edad)
    dataset['embarked'] = dataset['embarked'].fillna(moda_embarque)
    dataset['fare'] = dataset['fare'].fillna(mediana_fare)

# 6. One-Hot Encoding para las variables categóricas presentes
cols_categoricas = [c for c in ['sex', 'embarked', 'title', 'who'] if c in X_train_av.columns]
X_train_av = pd.get_dummies(X_train_av, columns=cols_categoricas, drop_first=True)
X_test_av = pd.get_dummies(X_test_av, columns=cols_categoricas, drop_first=True)
X_train_av, X_test_av = X_train_av.align(X_test_av, join='left', axis=1, fill_value=0)

# 7. Entrenar y comparar Árbol de Decisión vs. Random Forest
dt_model = DecisionTreeClassifier(max_depth=4, random_state=42)
dt_model.fit(X_train_av, y_train_av)
acc_dt = accuracy_score(y_test_av, dt_model.predict(X_test_av))

rf_model = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
rf_model.fit(X_train_av, y_train_av)
y_pred_rf = rf_model.predict(X_test_av)
acc_rf = accuracy_score(y_test_av, y_pred_rf)

print(f"Exactitud (Accuracy) Árbol de Decisión: {acc_dt:.4f}")
print(f"Exactitud (Accuracy) Random Forest:     {acc_rf:.4f}\n")
print("Reporte de Clasificación (Random Forest):")
print(classification_report(y_test_av, y_pred_rf))